In [1]:
import json
import asyncio
import aiohttp
import pandas as pd
from app.utils.path_util import get_project_root

# -----------------------------------------
# 0. Load labeled_ghost_domains.jsonl
# -----------------------------------------

input_path = get_project_root() / "notebooks/labeled_ghost_domains_training_data.jsonl"

raw_items = []
with open(input_path, "r", encoding="utf-8") as f:
    for line in f:
        obj = json.loads(line)
        # each line is {"domain": bool}
        domain, label_bool = next(iter(obj.items()))
        raw_items.append({"domain": domain, "label": int(label_bool)})

len(raw_items), raw_items[:3]



(40,
 [{'domain': 'rfacapitalcorp.com', 'label': 0},
  {'domain': 'rmiflooring.com', 'label': 0},
  {'domain': 'abcbni.com', 'label': 1}])

In [2]:
# from crawler.util.user_agents import USER_AGENTS
# import random
from notebooks.html_cleaner import clean_html
from crawler.util.url_normalizer import resolve_homepage

# -----------------------------------------
# Wrapper: fetch ONLY homepage HTML using production logic
# -----------------------------------------

async def fetch_html_via_orchestrator(session, site, timeout=10):
    """
    Uses resolve_homepage() exactly like crawl_domain() does,
    but returns ONLY the raw homepage HTML (cleaned).
    """
    base = site.rstrip("/") if site.startswith(("http://", "https://")) else f"https://{site}".rstrip("/")

    try:
        # user_agent = {"User-Agent": random.choice(USER_AGENTS)}
        homepage_info = await resolve_homepage(session, base, timeout, headers={})
        if homepage_info is None:
            return ""

        homepage_html, working_base = homepage_info

        # Clean semantic HTML for ML
        return clean_html(homepage_html)

    except Exception as e:
        print(f"[fetch_html_via_orchestrator] {domain} failed: {e}")
        return ""


# -----------------------------------------
# Fetch all suspicious sites using orchestrator logic
# -----------------------------------------

async def fetch_all(sites, timeout=10):
    connector = aiohttp.TCPConnector(limit=50, limit_per_host=5, ttl_dns_cache=600)

    async with aiohttp.ClientSession(connector=connector) as session:
        tasks = [fetch_html_via_orchestrator(session, d, timeout) for d in sites]
        return await asyncio.gather(*tasks)


In [3]:
# -----------------------------------------
# Fetch HTML for all domains
# -----------------------------------------

domains = [item["domain"] for item in raw_items]

import nest_asyncio
nest_asyncio.apply()

# !!!!!!!!!!!!!!!!
html_list = await fetch_all(domains) # !!!! Comment out this and comment below, PyCharm error is false positive
# html_list = fetch_all(domains) # Comment this
# !!!!!!!!!!!!!!!!

# -----------------------------------------
# Build dataframe
# -----------------------------------------

df = pd.DataFrame({
    "domain": domains,
    "html": html_list,
    "label": [item["label"] for item in raw_items],
})

df.head()

output_path = "labeled_ghost_domains.jsonl"

with open(output_path, "w", encoding="utf-8") as f:
    for _, row in df.iterrows():
        f.write(json.dumps({
            "domain": row["domain"],
            "html": row["html"],
            "label": int(row["label"]),
        }) + "\n")

df.head()


,domain,html,label
0,rfacapitalcorp.com,<title>Home - RFA Capital</title>\n<meta http-...,0
1,rmiflooring.com,<title>Flooring Contractors - RM Interiors</ti...,0
2,abcbni.com,<title>CharlesWorks web hosting web sites doma...,1
3,bhmedicalsupplies.com,<title>BH Supplies - Trusted Source for Syring...,0
4,whymessa.com,"<title>Why Messa &amp; Associates, P.C. — Pers...",0


In [4]:
# ============================================
# 2. Vectorize HTML using TF-IDF
# ============================================

# noinspection PyPackageRequirements
from sklearn.feature_extraction.text import TfidfVectorizer

# Keep vocabulary small → tiny inference module
MAX_FEATURES = 300

vectorizer = TfidfVectorizer(
    max_features=MAX_FEATURES,
    stop_words="english",
    ngram_range=(1, 2),
)

X = vectorizer.fit_transform(df["html"])
y = df["label"]

len(vectorizer.get_feature_names_out())


300

In [5]:
# ============================================
# 3. Train logistic regression
# ============================================

# noinspection PyPackageRequirements
from sklearn.linear_model import LogisticRegression

clf = LogisticRegression(max_iter=200)
clf.fit(X, y)

print("Training accuracy:", clf.score(X, y))


Training accuracy: 0.95


In [6]:
# ============================================
# 4. Export pure-Python inference module
# ============================================

feature_names = vectorizer.get_feature_names_out()
weights = dict(zip(feature_names, clf.coef_[0]))
bias = float(clf.intercept_[0])

export = {
    "bias": bias,
    "weights": weights,
}

classifier_weights_json = get_project_root() / "notebooks/ghost_classifier_weights.json"
with open(classifier_weights_json, "w", encoding="utf-8") as f:
    json.dump(export, f, indent=2)

print("Exported ghost_classifier_weights.json")


Exported ghost_classifier_weights.json


In [7]:
# ============================================
# Classify suspicious sites list
# ============================================

suspicious_path = get_project_root() / "notebooks/suspicious_sites.json"

# ============================================
# Load suspicious sites list
# ============================================

with open(suspicious_path, "r", encoding="utf-8") as f:
    suspicious_data = json.load(f)

new_domains = suspicious_data["data"]
len(new_domains), new_domains[:5]

# ============================================
# Fetch HTML for all suspicious domains
# ============================================

# !!!!!!!!!!!!!!!!
html_list = await fetch_all(new_domains) # !!!! Comment out this and comment below, PyCharm error is false positive
# html_list = fetch_all(new_domains) # Comment this
# !!!!!!!!!!!!!!!!

df = pd.DataFrame({
    "domain": new_domains,
    "html": html_list,
})

df.head()

# ============================================
# Classify using ghost_classifier.py
# ============================================

from ghost_classifier import is_ghost

df["ghost_score"] = df["html"].apply(lambda h: is_ghost(h, threshold=0.0))
df["ghost"] = df["ghost_score"].astype(bool)

df.head()

# ============================================
# Save classification results
# ============================================

new_output_path = get_project_root() / "notebooks/suspicious_sites_classified.jsonl"

with open(new_output_path, "w", encoding="utf-8") as f:
    for _, row in df.iterrows():
        f.write(json.dumps({
            "domain": row["domain"],
            "ghost": bool(row["ghost"]),
            "score": float(row["ghost_score"])
        }) + "\n")

new_output_path


WindowsPath('D:/DEV/Python/company-data-api/notebooks/suspicious_sites_classified.jsonl')